In [3]:
import os
import sys
from pathlib import Path
import pandas as pd
import json


In [ ]:


repo_id = "Exploration-Lab/iSign"
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not set")
SEED = 1990

ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

BUCKET = "isign_bucket"
GCS_PREFIX_dataset = "iSign/datasets"

word_presence_file = "word-presence-dataset_v1.1.csv"  # no trailing space
LOCAL_POSE_DIR = ROOT / "dataset/task4_word_presence/poses"
GCP_POSE_PREFIX = f"{BUCKET}/iSign/poses/iSign-poses_v1.1"

Project root: /Users/lokeshkumar/eMasters/EE964_Projects


In [7]:
from src.scripts.download_from_hugging_face import download_file_from_hf
# Download the dataset word-presence-dataset_v1.1.csv
download_file_from_hf(repo_id, "word-presence-dataset_v1.1.csv", ROOT / "dataset/task4_word_presence")


Total files in repo: 11
Downloaded: word-presence-dataset_v1.1.csv
Saved to: /Users/lokeshkumar/eMasters/EE964_Projects/dataset/task4_word_presence/word-presence-dataset_v1.1.csv


In [3]:
import pandas as pd
task4_df = pd.read_csv(ROOT / "dataset/task4_word_presence/word-presence-dataset_v1.1.csv")
task4_df.head()

,word_id,word,sentence_id,sentence
0,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e1,He was ashamed to admit to his mistake.
1,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e2,I felt deeply ashamed for my father's impolite...
2,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e3,I'm ashamed to be seen with you when you behav...
3,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e4,She was ashamed to ask her brother for money.
4,jYVESnkt5pc_w,veracity,jYVESnkt5pc_e1,Doubts were cast on the veracity of her alibi.


In [7]:
pose_files_to_download_from_gcp = []
for idx, row in task4_df.iterrows():
    word_file = row["word_id"]
    pose_files_to_download_from_gcp.append(word_file)
    sentence_file = row["sentence_id"]
    pose_files_to_download_from_gcp.append(sentence_file)
print("Number of pose files to download from GCP:", len(sorted(set(pose_files_to_download_from_gcp))))

Number of pose files to download from GCP: 2053


In [8]:
from src.scripts.download_from_GCP import download_pose_files_from_gcs
from src.scripts.authenticate_GCP import authenticate_gcp

# Authenticate with GCP
fs = authenticate_gcp(bucket_test="isign_bucket")

# download_pose_files_from_gcs(fs, LOCAL_POSE_DIR, POSE_PREFIX, df_clean=None, needed_ids: list = None):
download_pose_files_from_gcs(
    fs=fs,
    LOCAL_POSE_DIR=LOCAL_POSE_DIR,
    POSE_PREFIX=GCP_POSE_PREFIX,
    needed_ids=pose_files_to_download_from_gcp
)

GCS filesystem initialized (explicit ADC).
Total pose files needed: 2053


100%|██████████| 2053/2053 [1:09:38<00:00,  2.04s/it]

Downloaded now: 2038
Missing (404): 0
Forbidden (403/billing): 0


In [15]:
file_not_found = 0
total_downloaded_files = len(os.listdir(LOCAL_POSE_DIR))

for file in set(pose_files_to_download_from_gcp):
    if f"{file}.pose" not in os.listdir(LOCAL_POSE_DIR):
        file_not_found += 1

print(f"Total files not found after download: {file_not_found}")
print(f"Total downloaded files: {total_downloaded_files}")

Total files not found after download: 0
Total downloaded files: 2053


In [19]:
from pose_format import Pose

sample_pose_file = pose_files_to_download_from_gcp[0]
print("Sample pose file:", sample_pose_file)

with open(LOCAL_POSE_DIR / f"{sample_pose_file}.pose", "rb") as f:
    pose_features = Pose.read(f)
print("Pose features type:", type(pose_features))
print("Pose features shape:", numpy_data := pose_features.body.data.shape)

Sample pose file: U-c1RT4IfL8_w
Pose features type: <class 'pose_format.pose.Pose'>
Pose features shape: (143, 1, 576, 3)


In [20]:
print("Body fps:", pose_features.body.fps if hasattr(pose_features.body, "fps") else "N/A")
print("Body confidence shape:", getattr(pose_features.body, "confidence", None))
print("Body data dtype:", pose_features.body.data.dtype)
print("Min:", pose_features.body.data.min())
print("Max:", pose_features.body.data.max())

Body fps: 29
Body confidence shape: [[[9.9966705e-01 9.9941874e-01 9.9945623e-01 ... 1.2937213e-04
   3.4288180e-04 3.5822246e-04]]

 [[9.9962622e-01 9.9934191e-01 9.9938399e-01 ... 1.7301248e-04
   4.5920484e-04 4.7099125e-04]]

 [[9.9959999e-01 9.9930048e-01 9.9934542e-01 ... 2.3716746e-04
   5.8287015e-04 6.2055059e-04]]

 ...

 [[9.9864823e-01 9.9793965e-01 9.9816793e-01 ... 1.0132864e-03
   7.9913303e-04 1.2409668e-03]]

 [[9.9859077e-01 9.9784845e-01 9.9808186e-01 ... 1.0379246e-03
   8.1846298e-04 1.2763494e-03]]

 [[9.9855149e-01 9.9780536e-01 9.9806708e-01 ... 1.0840853e-03
   8.6031947e-04 1.2946134e-03]]]
Body data dtype: float32
Min: -204.40576
Max: 745.7429


In [21]:
import numpy as np

data = pose_features.body.data
print("NaN count:", np.isnan(data).sum())
print("Zero count:", np.sum(data == 0))

NaN count: 0
Zero count: 0


In [23]:
for sample_pose_file in pose_files_to_download_from_gcp[:5]:
    with open(LOCAL_POSE_DIR / f"{sample_pose_file}.pose", "rb") as f:
        pose = Pose.read(f)
    print(sample_pose_file, pose.body.data.shape)

U-c1RT4IfL8_w (143, 1, 576, 3)
U-c1RT4IfL8_e1 (251, 1, 576, 3)
U-c1RT4IfL8_w (143, 1, 576, 3)
U-c1RT4IfL8_e2 (251, 1, 576, 3)
U-c1RT4IfL8_w (143, 1, 576, 3)


In [27]:
x = pose.body.data

print("Pose data shape:", x.shape)

x = x.squeeze(1)  # remove person dimension
print("Pose data shape after squeeze:", x.shape)

x = x.reshape(x.shape[0], -1)  # flatten joints
print("Pose data shape after flatten:", x.shape)

Pose data shape: (143, 1, 576, 3)
Pose data shape after squeeze: (143, 576, 3)
Pose data shape after flatten: (143, 1728)


In [33]:
print(display(task4_df.head(10)))
print(task4_df.columns)
print("Unique word IDs:", task4_df["word_id"].unique()[:10])  # print first 10 unique word IDs
print("Unique sentence IDs:", task4_df["sentence_id"].unique()[:10])  # print first 10 unique sentence IDs
print("Total unique word IDs:", len(task4_df["word_id"].unique()))
print("Total unique sentence IDs:", len(task4_df["sentence_id"].unique()))
print("Total rows in dataset:", len(task4_df))

,word_id,word,sentence_id,sentence
0,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e1,He was ashamed to admit to his mistake.
1,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e2,I felt deeply ashamed for my father's impolite...
2,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e3,I'm ashamed to be seen with you when you behav...
3,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e4,She was ashamed to ask her brother for money.
4,jYVESnkt5pc_w,veracity,jYVESnkt5pc_e1,Doubts were cast on the veracity of her alibi.
5,jYVESnkt5pc_w,veracity,jYVESnkt5pc_e2,The chairman disallowed the veracity of his re...
6,jYVESnkt5pc_w,veracity,jYVESnkt5pc_e3,I don't doubt the veracity of your report.
7,jYVESnkt5pc_w,veracity,jYVESnkt5pc_e4,We have total confidence in the veracity of ou...
8,PcQgB9TbATk_w,maintenance,PcQgB9TbATk_e1,Old houses need a lot of maintenance.
9,J1xPH6Rugio_w,avail of,J1xPH6Rugio_e1,Guests are encouraged to avail themselves of t...


None
Index(['word_id', 'word', 'sentence_id', 'sentence'], dtype='object')
Unique word IDs: ['U-c1RT4IfL8_w' 'jYVESnkt5pc_w' 'PcQgB9TbATk_w' 'J1xPH6Rugio_w'
 'D9O50mQ7vRo_w' 'gCckaUKv4WM_w' 'c-PT1cEougQ_w' 'Miy-F4TYnso_w'
 'pmwPef49KGE_w' 'WX4SAl0w4fA_w']
Unique sentence IDs: ['U-c1RT4IfL8_e1' 'U-c1RT4IfL8_e2' 'U-c1RT4IfL8_e3' 'U-c1RT4IfL8_e4'
 'jYVESnkt5pc_e1' 'jYVESnkt5pc_e2' 'jYVESnkt5pc_e3' 'jYVESnkt5pc_e4'
 'PcQgB9TbATk_e1' 'J1xPH6Rugio_e1']
Total unique word IDs: 561
Total unique sentence IDs: 1492
Total rows in dataset: 1492


In [36]:
train_frac = 0.8
val_frac = 0.1
test_frac = 0.1

def get_base_name(file_id):
    return file_id.split("_")[0]

same_base = (task4_df["word_id"].apply(get_base_name) == task4_df["sentence_id"].apply(get_base_name))
print(same_base.value_counts())

True    1492
Name: count, dtype: int64


In [38]:
import numpy as np
task4_df["group_id"] = task4_df["word_id"].apply(get_base_name)
groups = sorted(task4_df["group_id"].unique())
rng = np.random.default_rng(SEED)
rng.shuffle(groups)

n = len(groups)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

train_groups = set(groups[:n_train])
val_groups   = set(groups[n_train:n_train+n_val])
test_groups  = set(groups[n_train+n_val:])

In [40]:
train_groups

{'-Gf2SGGyb4Y',
 '-W3Sefpg9l4',
 '-c5cUfu5gLU',
 '-du0Sljs6fg',
 '-jTNHftz4Pk',
 '0FUZJWoZ7Ho',
 '0YoN6sdJ84Y',
 '1A',
 '1k-AsMjTZbI',
 '1y33mnpgqyE',
 '21Toxytaq',
 '2A5tpFkt8ic',
 '2BPZs83dBQg',
 '2Fq4W-J5MZw',
 '2H-t8bYQGUE',
 '2LUrrdA633g',
 '2ldqLNKUekA',
 '2rrQeCAecbU',
 '30weefbWxWs',
 '3G3wXa7Ax',
 '3N7i4iMB4',
 '3hAYTF2TKGc',
 '3mMDY-zHiH8',
 '4QYv6IFjNbQ',
 '4RdjdI90j0M',
 '4uVjzBqNXkE',
 '4vfboqjtAXo',
 '50Qibhk4sD0',
 '5dfUuufFLZk',
 '5iIb7JC1mC8',
 '5ncS0vBocO0',
 '5xPTHpYQfeE',
 '68GdgKqKsrk',
 '69Vvb09Sq5U',
 '6DqUWmPTkHI',
 '6MxCP',
 '6egMMBnQUbY',
 '6mb7FlpNvFY',
 '6nwBq3QX9J0',
 '6qJDRpoe',
 '7PcZoqPll0k',
 '7S-pRkbzQ1I',
 '7tK-aXT9AM8',
 '861X7yZmXwU',
 '8Ui2qv0zkjA',
 '8WoE12Rn2SE',
 '9BKcG6j7LDU',
 '9OAMotf4jFE',
 '9pN1y7vMnEg',
 'ABZhHI6228g',
 'AHaC7ncPHjM',
 'AJbqUacnP-g',
 'AVDuEdEExK8',
 'AjXeuvRxnL0',
 'AnhTIMU',
 'B2xV05A7TbI',
 'B7idRBMZUvw',
 'BC1eDWKCSpQ',
 'BG',
 'BH7-iBjng',
 'BPu7NlTLvnU',
 'BWpU3Ya3iJo',
 'BYDhyNP5fK8',
 'BaSt1HrxXq0',
 'BsarjXadUR0',

In [41]:
train_df = task4_df[task4_df["group_id"].isin(train_groups)].copy()
val_df   = task4_df[task4_df["group_id"].isin(val_groups)].copy()
test_df  = task4_df[task4_df["group_id"].isin(test_groups)].copy()

In [42]:
assert set(train_df["group_id"]).isdisjoint(val_df["group_id"])
assert set(train_df["group_id"]).isdisjoint(test_df["group_id"])
assert set(val_df["group_id"]).isdisjoint(test_df["group_id"])

In [43]:
train_ids = set(train_df["word_id"]) | set(train_df["sentence_id"])
val_ids   = set(val_df["word_id"]) | set(val_df["sentence_id"])
test_ids  = set(test_df["word_id"]) | set(test_df["sentence_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

In [44]:
print("Train groups:", len(set(train_df["group_id"])))
print("Val groups:", len(set(val_df["group_id"])))
print("Test groups:", len(set(test_df["group_id"])))

print("Train ∩ Val:", len(set(train_df["group_id"]) & set(val_df["group_id"])))
print("Train ∩ Test:", len(set(train_df["group_id"]) & set(test_df["group_id"])))
print("Val ∩ Test:", len(set(val_df["group_id"]) & set(test_df["group_id"])))

Train groups: 388
Val groups: 83
Test groups: 84
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [45]:
train_ids = set(train_df["word_id"]) | set(train_df["sentence_id"])
val_ids   = set(val_df["word_id"]) | set(val_df["sentence_id"])
test_ids  = set(test_df["word_id"]) | set(test_df["sentence_id"])

print("Train ID count:", len(train_ids))
print("Val ID count:", len(val_ids))
print("Test ID count:", len(test_ids))

print("Train ∩ Val IDs:", len(train_ids & val_ids))
print("Train ∩ Test IDs:", len(train_ids & test_ids))
print("Val ∩ Test IDs:", len(val_ids & test_ids))

Train ID count: 1419
Val ID count: 341
Test ID count: 293
Train ∩ Val IDs: 0
Train ∩ Test IDs: 0
Val ∩ Test IDs: 0


In [46]:
assert len(set(train_df["group_id"]) & set(val_df["group_id"])) == 0
assert len(set(train_df["group_id"]) & set(test_df["group_id"])) == 0
assert len(set(val_df["group_id"]) & set(test_df["group_id"])) == 0

print("No group leakage across splits.")

No group leakage across splits.


In [47]:
print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Test rows:", len(test_df))

Train rows: 1030
Val rows: 253
Test rows: 209


In [49]:
split_dir = ROOT / "dataset/task4_word_presence/splits"

split_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(split_dir / "train.csv", index=False)
val_df.to_csv(split_dir / "val.csv", index=False)
test_df.to_csv(split_dir / "test.csv", index=False)

In [51]:
print("top 5 rows of train_df:")
display(train_df.head())
print("top 5 rows of val_df:")
display(val_df.head())
print("top 5 rows of test_df:")
display(test_df.head())

top 5 rows of train_df:


,word_id,word,sentence_id,sentence,group_id
0,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e1,He was ashamed to admit to his mistake.,U-c1RT4IfL8
1,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e2,I felt deeply ashamed for my father's impolite...,U-c1RT4IfL8
2,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e3,I'm ashamed to be seen with you when you behav...,U-c1RT4IfL8
3,U-c1RT4IfL8_w,ashamed,U-c1RT4IfL8_e4,She was ashamed to ask her brother for money.,U-c1RT4IfL8
4,jYVESnkt5pc_w,veracity,jYVESnkt5pc_e1,Doubts were cast on the veracity of her alibi.,jYVESnkt5pc


top 5 rows of val_df:


,word_id,word,sentence_id,sentence,group_id
13,D9O50mQ7vRo_w,bestow,D9O50mQ7vRo_e1,The Queen has bestowed a knighthood on him.,D9O50mQ7vRo
14,D9O50mQ7vRo_w,bestow,D9O50mQ7vRo_e2,The university bestowed an honorary degree upo...,D9O50mQ7vRo
15,D9O50mQ7vRo_w,bestow,D9O50mQ7vRo_e3,"During the ceremony, the prime minister will b...",D9O50mQ7vRo
16,D9O50mQ7vRo_w,bestow,D9O50mQ7vRo_e4,The country's highest medal was bestowed upon ...,D9O50mQ7vRo
33,WX4SAl0w4fA_w,embarrassed,WX4SAl0w4fA_e1,He was so embarrassed - his face went brick-red .,WX4SAl0w4fA


top 5 rows of test_df:


,word_id,word,sentence_id,sentence,group_id
29,pmwPef49KGE_w,convivial,pmwPef49KGE_e1,The mood was relaxed and convivial.,pmwPef49KGE
30,pmwPef49KGE_w,convivial,pmwPef49KGE_e2,"Dury is convivial, affable and engaging.",pmwPef49KGE
31,pmwPef49KGE_w,convivial,pmwPef49KGE_e3,The atmosphere was quite convivial.,pmwPef49KGE
32,pmwPef49KGE_w,convivial,pmwPef49KGE_e4,"Eat slowly in convivial surroundings and, abov...",pmwPef49KGE
38,scJSFdl6vqk_w,quandary,scJSFdl6vqk_e1,"I've had two job offers, and I'm in a real qua...",scJSFdl6vqk


In [52]:
from pathlib import Path
import pandas as pd
import numpy as np

SEED = 1990
NEG_PER_POS = 3

split_dir = ROOT / "dataset/task4_word_presence/splits"

train_pos = pd.read_csv(split_dir / "train.csv")
val_pos   = pd.read_csv(split_dir / "val.csv")
test_pos  = pd.read_csv(split_dir / "test.csv")


In [53]:
from src.utils.pair_generation import build_labeled_pairs
train_pairs = build_labeled_pairs(train_pos, neg_per_pos=NEG_PER_POS, seed=SEED)
val_pairs   = build_labeled_pairs(val_pos, neg_per_pos=NEG_PER_POS, seed=SEED)
test_pairs  = build_labeled_pairs(test_pos, neg_per_pos=NEG_PER_POS, seed=SEED)
print("Train pairs shape:", train_pairs.shape)
print("Val pairs shape:", val_pairs.shape)
print("Test pairs shape:", test_pairs.shape)

Train pairs shape: (4120, 6)
Val pairs shape: (1012, 6)
Test pairs shape: (836, 6)


In [54]:
train_pairs.to_csv(split_dir / "train_labeled.csv", index=False)
val_pairs.to_csv(split_dir / "val_labeled.csv", index=False)
test_pairs.to_csv(split_dir / "test_labeled.csv", index=False)

print("Train labeled:", train_pairs.shape)
print(train_pairs["label"].value_counts())

print("Val labeled:", val_pairs.shape)
print(val_pairs["label"].value_counts())

print("Test labeled:", test_pairs.shape)
print(test_pairs["label"].value_counts())

Train labeled: (4120, 6)
label
0    3090
1    1030
Name: count, dtype: int64
Val labeled: (1012, 6)
label
0    759
1    253
Name: count, dtype: int64
Test labeled: (836, 6)
label
0    627
1    209
Name: count, dtype: int64


In [55]:
print("Train negatives use only train sentence IDs:",
      set(train_pairs["sentence_id"]).issubset(set(train_pos["sentence_id"])))

print("Val negatives use only val sentence IDs:",
      set(val_pairs["sentence_id"]).issubset(set(val_pos["sentence_id"])))

print("Test negatives use only test sentence IDs:",
      set(test_pairs["sentence_id"]).issubset(set(test_pos["sentence_id"])))

Train negatives use only train sentence IDs: True
Val negatives use only val sentence IDs: True
Test negatives use only test sentence IDs: True


In [56]:
def check_no_false_negatives(pos_df, labeled_df):
    pos_pairs = set(zip(pos_df["word_id"], pos_df["sentence_id"]))
    neg_pairs = set(zip(
        labeled_df.loc[labeled_df["label"] == 0, "word_id"],
        labeled_df.loc[labeled_df["label"] == 0, "sentence_id"]
    ))
    print("False negative overlaps:", len(pos_pairs & neg_pairs))

check_no_false_negatives(train_pos, train_pairs)
check_no_false_negatives(val_pos, val_pairs)
check_no_false_negatives(test_pos, test_pairs)

False negative overlaps: 0
False negative overlaps: 0
False negative overlaps: 0


In [57]:
from src.utils.pose_utils import load_and_normalize_pose

x = load_and_normalize_pose(LOCAL_POSE_DIR / f"{sample_pose_file}.pose")
print(x.shape)
print(x.mean(), x.std())

(143, 1728)
2.485935570913997e-08 0.999753589633966


In [58]:
from src.utils.pose_utils import load_pose_as_array, load_and_normalize_pose

raw_x = load_pose_as_array(LOCAL_POSE_DIR / f"{sample_pose_file}.pose")
norm_x = load_and_normalize_pose(LOCAL_POSE_DIR / f"{sample_pose_file}.pose")

print("Raw shape:", raw_x.shape)
print("Norm shape:", norm_x.shape)

print("Raw min/max:", raw_x.min(), raw_x.max())
print("Norm min/max:", norm_x.min(), norm_x.max())

print("Norm mean/std:", norm_x.mean(), norm_x.std())

Raw shape: (143, 1728)
Norm shape: (143, 1728)
Raw min/max: -204.40576 745.7429
Norm min/max: -4.6100407 3.6359274
Norm mean/std: 2.485935570913997e-08 0.999753589633966


In [60]:
from src.tasks.task4_word_presence.data.dataset import Task4WordPresenceDataset

ds = Task4WordPresenceDataset(
    split_csv=ROOT / "dataset/task4_word_presence/splits/train_labeled.csv",
    pose_dir=ROOT / "/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task4_word_presence/poses",
)

sample = ds[0]
print(sample["word_id"], sample["sentence_id"], sample["label"])
print(sample["word_x"].shape, sample["sent_x"].shape)

yxme3eH9KiI_w yxme3eH9KiI_e2 tensor(1.)
torch.Size([433, 1728]) torch.Size([191, 1728])


In [62]:
from torch.utils.data import DataLoader
from src.tasks.task4_word_presence.data.dataset import (
    Task4WordPresenceDataset,
    task4_collate_fn,
)

ds = Task4WordPresenceDataset(
    split_csv=ROOT / "dataset/task4_word_presence/splits/train_labeled.csv",
    pose_dir=ROOT / "dataset/task4_word_presence/poses",
)

loader = DataLoader(
    ds,
    batch_size=4,
    shuffle=False,
    collate_fn=task4_collate_fn,
)

batch = next(iter(loader))

print(batch["word_x"].shape)
print(batch["word_len"])
print(batch["sent_x"].shape)
print(batch["sent_len"])
print(batch["label"].shape)
print(batch["label"])

torch.Size([4, 433, 1728])
tensor([433, 153, 182, 183])
torch.Size([4, 470, 1728])
tensor([191, 171, 350, 470])
torch.Size([4])
tensor([1., 0., 0., 0.])


In [63]:
from src.tasks.task4_word_presence.models.stat_pool_baseline import StatPoolBaseline

model = StatPoolBaseline(
    input_dim=1728,
    hidden_dim=256,
    output_dim=256,
    pool="mean",
    dropout=0.1,
)

logits, cosine = model(
    batch["word_x"],
    batch["word_len"],
    batch["sent_x"],
    batch["sent_len"],
)

print("logits shape:", logits.shape)
print("cosine shape:", cosine.shape)
print("logits:", logits)
print("cosine:", cosine)

logits shape: torch.Size([4])
cosine shape: torch.Size([4])
logits: tensor([8.7936, 7.9853, 8.6970, 8.5190], grad_fn=<AddBackward0>)
cosine: tensor([0.8794, 0.7985, 0.8697, 0.8519], grad_fn=<SumBackward1>)


In [64]:
from src.tasks.task4_word_presence.models.stat_pool_baseline import StatPoolBaseline
from src.tasks.task4_word_presence.training.train_word_presence import run_training
from src.utils.set_seeds import set_seed

set_seed(1990)

model = StatPoolBaseline(
    input_dim=1728,
    hidden_dim=256,
    output_dim=256,
    pool="mean",
    dropout=0.1,
)

results = run_training(
    train_csv=ROOT / "dataset/task4_word_presence/splits/train_labeled.csv",
    val_csv=ROOT / "dataset/task4_word_presence/splits/val_labeled.csv",
    test_csv=ROOT / "dataset/task4_word_presence/splits/test_labeled.csv",
    pose_dir=ROOT / "dataset/task4_word_presence/poses",
    out_dir=ROOT / "results/task4_word_presence/debug_statpool_mean",
    batch_size=8,
    num_workers=0,
    lr=2e-4,
    weight_decay=1e-4,
    epochs=2,
    early_stopping_patience=None,
    model=model,
)

Epoch 01 | train_loss=1.5841 | train_acc=53.23 | val_loss=1.2675 | val_acc=56.52
Epoch 02 | train_loss=0.9082 | train_acc=63.47 | val_loss=0.9406 | val_acc=65.51

Best validation accuracy: 65.51383399209486
Test metrics: {'loss': 0.897853828289292, 'accuracy': 66.02870813397129}


In [65]:
from src.tasks.task4_word_presence.data.precompute_pose_npy import precompute_pose_npy

summary = precompute_pose_npy(
    csv_paths=[
        ROOT / "dataset/task4_word_presence/splits/train_labeled.csv",
        ROOT / "dataset/task4_word_presence/splits/val_labeled.csv",
        ROOT / "dataset/task4_word_presence/splits/test_labeled.csv",
    ],
    pose_dir=ROOT / "dataset/task4_word_presence/poses",
    out_dir=ROOT / "dataset/task4_word_presence/pose_npy",
    overwrite=False,
)

Precomputing pose npy: 100%|██████████| 2053/2053 [00:21<00:00, 93.44it/s] 

{
  "num_total_ids": 2053,
  "num_done": 2053,
  "num_skipped": 0,
  "num_failed": 0,
  "failures": []
}


In [3]:
from src.tasks.task4_word_presence.data.dataset import Task4WordPresenceDataset

ds = Task4WordPresenceDataset(
    split_csv=ROOT / "dataset/task4_word_presence/splits/train_labeled.csv",
    feature_dir=ROOT / "dataset/task4_word_presence/pose_npy",
)

sample = ds[0]
print(sample["word_id"], sample["sentence_id"], sample["label"])
print(sample["word_x"].shape, sample["sent_x"].shape)
print(sample["word_x"].dtype, sample["sent_x"].dtype)

yxme3eH9KiI_w yxme3eH9KiI_e2 tensor(1.)
torch.Size([433, 1728]) torch.Size([191, 1728])
torch.float32 torch.float32


In [4]:
from src.utils.set_seeds import set_seed
from src.utils.expand_grid import expand_grid
from src.tasks.task4_word_presence.models.stat_pool_baseline import StatPoolBaseline
from src.tasks.task4_word_presence.training.train_word_presence import run_training

import os
import pandas as pd

seeds = [1990]   # first verify with one seed; later add 42, 2024

StatPoolBaseline_grid_stage0 = {
    "input_dim": [1728],
    "hidden_dim": [256],
    "output_dim": [256],
    "dropout": [0.0, 0.1],
    "lr": [2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "pool": ["mean", "max", "meanmax", "meanstd"],
}

NUM_EPOCHS = 20
patience = 5

results_csv_StatPoolBaseline_with_seeds = (
    "results/task4_word_presence/statpool_gridsearch_results_with_seeds.csv"
)

configs_StatPoolBaseline = expand_grid(StatPoolBaseline_grid_stage0)

print(f"Total configs: {len(configs_StatPoolBaseline)}")
print("Sample config:", configs_StatPoolBaseline[0])

all_results = []

for seed in seeds:
    print("\n" + "=" * 80)
    print(f"Running seed: {seed}")
    set_seed(seed)

    for i, cfg in enumerate(configs_StatPoolBaseline, start=1):
        print("\n" + "=" * 80)
        print(f"[{i}/{len(configs_StatPoolBaseline)}] Running StatPoolBaseline with config:")
        print(cfg)

        exp_name = (
            f"statpool_"
            f"pool{cfg['pool']}_"
            f"h{cfg['hidden_dim']}_o{cfg['output_dim']}_"
            f"drop{str(cfg['dropout']).replace('.', 'p')}_"
            f"lr{cfg['lr']}_"
            f"wd{cfg['weight_decay']}_"
            f"seed{seed}"
        )

        try:
            model = StatPoolBaseline(
                input_dim=cfg["input_dim"],
                hidden_dim=cfg["hidden_dim"],
                output_dim=cfg["output_dim"],
                pool=cfg["pool"],
                dropout=cfg["dropout"],
            )

            results = run_training(
                train_csv=ROOT / "dataset/task4_word_presence/splits/train_labeled.csv",
                val_csv=ROOT / "dataset/task4_word_presence/splits/val_labeled.csv",
                test_csv=ROOT / "dataset/task4_word_presence/splits/test_labeled.csv",
                feature_dir=ROOT / "dataset/task4_word_presence/pose_npy",
                out_dir=ROOT / "results/task4_word_presence/StatPoolBaseline_gridsearch" / exp_name,
                batch_size=cfg["batch_size"],
                num_workers=0,
                lr=cfg["lr"],
                weight_decay=cfg["weight_decay"],
                epochs=NUM_EPOCHS,
                early_stopping_patience=patience,
                model=model,
            )

            val_rank_str = (
            f"{results['val_rank_metrics']['avg_rank']:.1f}/"
            f"{results['val_rank_metrics']['num_candidates']}"
            )

            test_rank_str = (
            f"{results['test_rank_metrics']['avg_rank']:.1f}/"
            f"{results['test_rank_metrics']['num_candidates']}"
            )

            row = {
                "exp_name": exp_name,
                **cfg,
                "best_val_accuracy": results["best_val_accuracy"],
                "val_accuracy": results["best_val_metrics"]["accuracy"],
                "val_loss": results["best_val_metrics"]["loss"],
                "val_avg_rank": results["val_rank_metrics"]["avg_rank"],
                "val_num_candidates": results["val_rank_metrics"]["num_candidates"],
                "val_rank_str": val_rank_str,
                "test_accuracy": results["test_metrics"]["accuracy"],
                "test_loss": results["test_metrics"]["loss"],
                "test_avg_rank": results["test_rank_metrics"]["avg_rank"],
                "test_num_candidates": results["test_rank_metrics"]["num_candidates"],
                "test_rank_str": test_rank_str,
                "seed": seed,
            }

        except Exception as e:
            print(f"Failed config: {cfg}")
            print("Error:", e)
            row = {
                "exp_name": exp_name,
                **cfg,
                "seed": seed,
                "error": str(e),
            }

        all_results.append(row)

        pd.DataFrame([row]).to_csv(
            ROOT / results_csv_StatPoolBaseline_with_seeds,
            mode="a",
            header=not os.path.exists(ROOT / results_csv_StatPoolBaseline_with_seeds),
            index=False,
        )

df = pd.DataFrame(all_results)
display(df.sort_values(["test_accuracy", "test_avg_rank"], ascending=[False, True]))

Total configs: 8
Sample config: {'input_dim': 1728, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'pool': 'mean'}

Running seed: 1990

[1/8] Running StatPoolBaseline with config:
{'input_dim': 1728, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'pool': 'mean'}
Epoch 01 | train_loss=1.7229 | train_acc=54.20 | val_loss=1.0626 | val_acc=57.31 | val_avg_rank=30.81
Epoch 02 | train_loss=0.8533 | train_acc=64.90 | val_loss=0.8144 | val_acc=65.12 | val_avg_rank=27.90
Epoch 03 | train_loss=0.6673 | train_acc=71.00 | val_loss=0.7422 | val_acc=67.69 | val_avg_rank=27.12
Epoch 04 | train_loss=0.5577 | train_acc=74.76 | val_loss=0.7552 | val_acc=66.50
Epoch 05 | train_loss=0.5281 | train_acc=75.78 | val_loss=0.8238 | val_acc=62.35
Epoch 06 | train_loss=0.4676 | train_acc=78.18 | val_loss=0.6892 | val_acc=67.49
Epoch 07 | train_loss=0.4225 | train_acc=80.90 | val_loss=0.68

,exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,pool,best_val_accuracy,...,val_loss,val_avg_rank,val_num_candidates,val_rank_str,test_accuracy,test_loss,test_avg_rank,test_num_candidates,test_rank_str,seed
3,statpool_poolmeanstd_h256_o256_drop0p0_lr0.000...,1728,256,256,0.0,0.0002,0.0001,32,meanstd,76.679842,...,0.515388,16.159091,253,16.2/253,78.468900,0.479336,18.071429,209,18.1/209,1990
2,statpool_poolmeanmax_h256_o256_drop0p0_lr0.000...,1728,256,256,0.0,0.0002,0.0001,32,meanmax,76.581028,...,0.520560,17.636364,253,17.6/253,75.956938,0.521816,20.250000,209,20.2/209,1990
7,statpool_poolmeanstd_h256_o256_drop0p1_lr0.000...,1728,256,256,0.1,0.0002,0.0001,32,meanstd,72.134387,...,0.630161,19.420455,253,19.4/253,75.000000,0.544906,16.452381,209,16.5/209,1990
5,statpool_poolmax_h256_o256_drop0p1_lr0.0002_wd...,1728,256,256,0.1,0.0002,0.0001,32,max,71.640316,...,0.626570,24.625000,253,24.6/253,74.162679,0.573887,24.214286,209,24.2/209,1990
0,statpool_poolmean_h256_o256_drop0p0_lr0.0002_w...,1728,256,256,0.0,0.0002,0.0001,32,mean,73.715415,...,0.553207,16.261364,253,16.3/253,73.803828,0.583567,12.416667,209,12.4/209,1990
6,statpool_poolmeanmax_h256_o256_drop0p1_lr0.000...,1728,256,256,0.1,0.0002,0.0001,32,meanmax,71.541502,...,0.625194,22.920455,253,22.9/253,73.086124,0.589925,24.833333,209,24.8/209,1990
4,statpool_poolmean_h256_o256_drop0p1_lr0.0002_w...,1728,256,256,0.1,0.0002,0.0001,32,mean,71.146245,...,0.660429,21.397727,253,21.4/253,72.009569,0.660645,16.690476,209,16.7/209,1990
1,statpool_poolmax_h256_o256_drop0p0_lr0.0002_wd...,1728,256,256,0.0,0.0002,0.0001,32,max,70.158103,...,0.628536,27.670455,253,27.7/253,71.172249,0.587506,27.273810,209,27.3/209,1990


In [8]:
import pandas as pd

df = pd.read_csv(
    "/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence/statpool_gridsearch_results_with_seeds.csv",
    engine="python",
    on_bad_lines="skip"   # skip malformed rows (20-column ones)
)

# remove first 8 rows of old seed 1990 runs
df = df.iloc[8:].copy()

In [10]:
import pandas as pd
from io import StringIO

csv_path = "/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence/statpool_gridsearch_results_with_seeds.csv"

with open(csv_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# keep original header line separately
header = lines[0]

# remove first 8 old runs after header
remaining_lines = [header] + lines[9:]

In [11]:
valid_21 = []

for line in remaining_lines[1:]:
    if len(line.strip().split(",")) == 21:
        valid_21.append(line)

print("Final valid rows:", len(valid_21))

Final valid rows: 24


In [12]:
final_columns = [
    "exp_name",
    "input_dim",
    "hidden_dim",
    "output_dim",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "pool",
    "best_val_accuracy",
    "val_accuracy",
    "val_loss",
    "val_avg_rank",
    "val_num_queries",
    "val_rank_str",
    "test_accuracy",
    "test_loss",
    "test_avg_rank",
    "test_num_queries",
    "test_rank_str",
    "seed",
]

df = pd.read_csv(
    StringIO("".join(valid_21)),
    header=None,
    names=final_columns
)

print(df.head())
print(df.shape)
print(df.columns.tolist())

                                            exp_name  input_dim  hidden_dim  \
0  statpool_poolmean_h256_o256_drop0p0_lr0.0002_w...       1728         256   
1  statpool_poolmax_h256_o256_drop0p0_lr0.0002_wd...       1728         256   
2  statpool_poolmeanmax_h256_o256_drop0p0_lr0.000...       1728         256   
3  statpool_poolmeanstd_h256_o256_drop0p0_lr0.000...       1728         256   
4  statpool_poolmean_h256_o256_drop0p1_lr0.0002_w...       1728         256   

   output_dim  dropout      lr  weight_decay  batch_size     pool  \
0         256      0.0  0.0002        0.0001          32     mean   
1         256      0.0  0.0002        0.0001          32      max   
2         256      0.0  0.0002        0.0001          32  meanmax   
3         256      0.0  0.0002        0.0001          32  meanstd   
4         256      0.1  0.0002        0.0001          32     mean   

   best_val_accuracy  ...  val_loss  val_avg_rank  val_num_queries  \
0          73.616601  ...  0.585015     

In [13]:
df = df.rename(columns={
    "best_val_accuracy": "val_acc",
    "test_accuracy": "test_acc",
    "test_avg_rank": "test_rank"
})

df = df[[
    "pool",
    "dropout",
    "seed",
    "val_acc",
    "test_acc",
    "test_rank"
]]

print(df.head())

      pool  dropout  seed    val_acc   test_acc  test_rank
0     mean      0.0    42  73.616601  76.913876  12.702381
1      max      0.0    42  75.988142  76.076555  20.273810
2  meanmax      0.0    42  73.814229  77.511962  19.321429
3  meanstd      0.0    42  74.901186  77.631579  16.357143
4     mean      0.1    42  74.209486  77.272727  12.797619


In [14]:
agg = (
    df
    .groupby(["pool", "dropout"])
    .agg(
        val_acc_mean=("val_acc", "mean"),
        val_acc_std=("val_acc", "std"),
        test_acc_mean=("test_acc", "mean"),
        test_acc_std=("test_acc", "std"),
        test_rank_mean=("test_rank", "mean"),
        test_rank_std=("test_rank", "std"),
    )
    .reset_index()
)

agg = agg.round(2)
print(agg)

      pool  dropout  val_acc_mean  val_acc_std  test_acc_mean  test_acc_std  \
0      max      0.0         73.81         3.19          74.72          3.10   
1      max      0.1         74.14         2.74          75.08          1.91   
2     mean      0.0         73.52         0.26          74.24          2.48   
3     mean      0.1         72.63         1.53          74.80          2.65   
4  meanmax      0.0         74.97         1.44          76.56          0.84   
5  meanmax      0.1         73.22         1.48          73.60          0.70   
6  meanstd      0.0         74.80         1.93          77.23          1.48   
7  meanstd      0.1         73.48         1.29          76.95          1.86   

   test_rank_mean  test_rank_std  
0           23.00           3.75  
1           21.88           2.85  
2           13.56           1.73  
3           14.55           1.98  
4           20.87           1.93  
5           22.55           2.32  
6           18.11           1.77  
7       

In [15]:
import pandas as pd
from io import StringIO

csv_path = "/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence/statpool_gridsearch_results_with_seeds.csv"

final_columns = [
    "exp_name",
    "input_dim",
    "hidden_dim",
    "output_dim",
    "dropout",
    "lr",
    "weight_decay",
    "batch_size",
    "pool",
    "best_val_accuracy",
    "val_accuracy",
    "val_loss",
    "val_avg_rank",
    "val_num_queries",
    "val_rank_str",
    "test_accuracy",
    "test_loss",
    "test_avg_rank",
    "test_num_queries",
    "test_rank_str",
    "seed",
]

with open(csv_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# remove first 8 old runs, then keep only final 21-column rows
valid_21 = []
for line in lines[9:]:
    if len(line.strip().split(",")) == 21:
        valid_21.append(line)

df = pd.read_csv(
    StringIO("".join(valid_21)),
    header=None,
    names=final_columns
)

df = df.rename(columns={
    "best_val_accuracy": "val_acc",
    "test_accuracy": "test_acc",
    "test_avg_rank": "test_rank"
})

# keep full config identity
df = df[
    [
        "input_dim",
        "hidden_dim",
        "output_dim",
        "dropout",
        "lr",
        "weight_decay",
        "batch_size",
        "pool",
        "seed",
        "val_acc",
        "test_acc",
        "test_rank",
    ]
].copy()

print(df.head())

   input_dim  hidden_dim  output_dim  dropout      lr  weight_decay  \
0       1728         256         256      0.0  0.0002        0.0001   
1       1728         256         256      0.0  0.0002        0.0001   
2       1728         256         256      0.0  0.0002        0.0001   
3       1728         256         256      0.0  0.0002        0.0001   
4       1728         256         256      0.1  0.0002        0.0001   

   batch_size     pool  seed    val_acc   test_acc  test_rank  
0          32     mean    42  73.616601  76.913876  12.702381  
1          32      max    42  75.988142  76.076555  20.273810  
2          32  meanmax    42  73.814229  77.511962  19.321429  
3          32  meanstd    42  74.901186  77.631579  16.357143  
4          32     mean    42  74.209486  77.272727  12.797619  


In [17]:
agg = (
    df.groupby(
        [
            "input_dim",
            "hidden_dim",
            "output_dim",
            "dropout",
            "lr",
            "weight_decay",
            "batch_size",
            "pool",
        ],
        as_index=False
    )
    .agg(
        runs=("seed", "count"),
        seeds=("seed", lambda x: sorted(x.tolist())),
        val_acc_mean=("val_acc", "mean"),
        val_acc_std=("val_acc", "std"),
        test_acc_mean=("test_acc", "mean"),
        test_acc_std=("test_acc", "std"),
        test_rank_mean=("test_rank", "mean"),
        test_rank_std=("test_rank", "std"),
    )
)

agg = agg.round({
    "val_acc_mean": 2,
    "val_acc_std": 2,
    "test_acc_mean": 2,
    "test_acc_std": 2,
    "test_rank_mean": 2,
    "test_rank_std": 2,
})

display(agg.sort_values(["test_acc_mean", "test_rank_mean"], ascending=[False, True]))

,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,pool,runs,seeds,val_acc_mean,val_acc_std,test_acc_mean,test_acc_std,test_rank_mean,test_rank_std
3,1728,256,256,0.0,0.0002,0.0001,32,meanstd,3,"[42, 1990, 2024]",74.80,1.93,77.23,1.48,18.11,1.77
7,1728,256,256,0.1,0.0002,0.0001,32,meanstd,3,"[42, 1990, 2024]",73.48,1.29,76.95,1.86,16.02,0.69
2,1728,256,256,0.0,0.0002,0.0001,32,meanmax,3,"[42, 1990, 2024]",74.97,1.44,76.56,0.84,20.87,1.93
4,1728,256,256,0.1,0.0002,0.0001,32,max,3,"[42, 1990, 2024]",74.14,2.74,75.08,1.91,21.88,2.85
5,1728,256,256,0.1,0.0002,0.0001,32,mean,3,"[42, 1990, 2024]",72.63,1.53,74.80,2.65,14.55,1.98
0,1728,256,256,0.0,0.0002,0.0001,32,max,3,"[42, 1990, 2024]",73.81,3.19,74.72,3.10,23.00,3.75
1,1728,256,256,0.0,0.0002,0.0001,32,mean,3,"[42, 1990, 2024]",73.52,0.26,74.24,2.48,13.56,1.73
6,1728,256,256,0.1,0.0002,0.0001,32,meanmax,3,"[42, 1990, 2024]",73.22,1.48,73.60,0.70,22.55,2.32


In [18]:
best_acc = agg.sort_values(
    by=["test_acc_mean", "test_rank_mean"],
    ascending=[False, True]
).reset_index(drop=True)

print(best_acc)

   input_dim  hidden_dim  output_dim  dropout      lr  weight_decay  \
0       1728         256         256      0.0  0.0002        0.0001   
1       1728         256         256      0.1  0.0002        0.0001   
2       1728         256         256      0.0  0.0002        0.0001   
3       1728         256         256      0.1  0.0002        0.0001   
4       1728         256         256      0.1  0.0002        0.0001   
5       1728         256         256      0.0  0.0002        0.0001   
6       1728         256         256      0.0  0.0002        0.0001   
7       1728         256         256      0.1  0.0002        0.0001   

   batch_size     pool  runs             seeds  val_acc_mean  val_acc_std  \
0          32  meanstd     3  [42, 1990, 2024]         74.80         1.93   
1          32  meanstd     3  [42, 1990, 2024]         73.48         1.29   
2          32  meanmax     3  [42, 1990, 2024]         74.97         1.44   
3          32      max     3  [42, 1990, 2024]      

In [19]:
best_rank = agg.sort_values(
    by=["test_rank_mean", "test_acc_mean"],
    ascending=[True, False]
).reset_index(drop=True)

print(best_rank)

   input_dim  hidden_dim  output_dim  dropout      lr  weight_decay  \
0       1728         256         256      0.0  0.0002        0.0001   
1       1728         256         256      0.1  0.0002        0.0001   
2       1728         256         256      0.1  0.0002        0.0001   
3       1728         256         256      0.0  0.0002        0.0001   
4       1728         256         256      0.0  0.0002        0.0001   
5       1728         256         256      0.1  0.0002        0.0001   
6       1728         256         256      0.1  0.0002        0.0001   
7       1728         256         256      0.0  0.0002        0.0001   

   batch_size     pool  runs             seeds  val_acc_mean  val_acc_std  \
0          32     mean     3  [42, 1990, 2024]         73.52         0.26   
1          32     mean     3  [42, 1990, 2024]         72.63         1.53   
2          32  meanstd     3  [42, 1990, 2024]         73.48         1.29   
3          32  meanstd     3  [42, 1990, 2024]      

In [20]:
report = agg.copy()

report["val_acc"] = report["val_acc_mean"].astype(str) + " ± " + report["val_acc_std"].astype(str)
report["test_acc"] = report["test_acc_mean"].astype(str) + " ± " + report["test_acc_std"].astype(str)
report["test_rank"] = report["test_rank_mean"].astype(str) + " ± " + report["test_rank_std"].astype(str)

report = report[
    [
        "input_dim",
        "hidden_dim",
        "output_dim",
        "dropout",
        "lr",
        "weight_decay",
        "batch_size",
        "pool",
        "runs",
        "seeds",
        "val_acc",
        "test_acc",
        "test_rank",
    ]
]

print(report)

   input_dim  hidden_dim  output_dim  dropout      lr  weight_decay  \
0       1728         256         256      0.0  0.0002        0.0001   
1       1728         256         256      0.0  0.0002        0.0001   
2       1728         256         256      0.0  0.0002        0.0001   
3       1728         256         256      0.0  0.0002        0.0001   
4       1728         256         256      0.1  0.0002        0.0001   
5       1728         256         256      0.1  0.0002        0.0001   
6       1728         256         256      0.1  0.0002        0.0001   
7       1728         256         256      0.1  0.0002        0.0001   

   batch_size     pool  runs             seeds       val_acc      test_acc  \
0          32      max     3  [42, 1990, 2024]  73.81 ± 3.19   74.72 ± 3.1   
1          32     mean     3  [42, 1990, 2024]  73.52 ± 0.26  74.24 ± 2.48   
2          32  meanmax     3  [42, 1990, 2024]  74.97 ± 1.44  76.56 ± 0.84   
3          32  meanstd     3  [42, 1990, 2024]  

### Phase-1


In [5]:
seeds = [1990]

GRU_grid_stage1_smoke = {
    "encoder_type": ["gru", "bigru"],
    "input_dim": [1728],
    "hidden_dim": [256],
    "output_dim": [256],
    "dropout": [0.0],
    "lr": [2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "pool": ["mean", "meanstd"],
    "num_layers": [1],
}

NUM_EPOCHS = 2
patience = 2

In [6]:
from src.utils.set_seeds import set_seed
from src.utils.expand_grid import expand_grid
from src.tasks.task4_word_presence.models.gru_models import (
    GRUEncoder,
    BiGRUEncoder,
    WordPresenceGRUModel,
)
from src.tasks.task4_word_presence.training.train_word_presence import run_training

import os
import pandas as pd

seeds = [1990]   # smoke first; later [1990, 42, 2024]

GRU_grid_stage1 = {
    "encoder_type": ["gru", "bigru"],
    "input_dim": [1728],
    "hidden_dim": [256],
    "output_dim": [256],
    "dropout": [0.0, 0.1],
    "lr": [2e-4],
    "weight_decay": [1e-4],
    "batch_size": [32],
    "pool": ["mean", "meanstd"],
    "num_layers": [1],
}

NUM_EPOCHS = 2
patience = 2

results_csv_GRU_with_seeds = (
    "results/task4_word_presence/gru_gridsearch_results_with_seeds_smoke_test.csv"
)

configs_GRU = expand_grid(GRU_grid_stage1_smoke)

print(f"Total configs: {len(configs_GRU)}")
print("Sample config:", configs_GRU[0])

all_results = []

for seed in seeds:
    print("\n" + "=" * 80)
    print(f"Running seed: {seed}")
    set_seed(seed)

    for i, cfg in enumerate(configs_GRU, start=1):
        print("\n" + "=" * 80)
        print(f"[{i}/{len(configs_GRU)}] Running GRU model with config:")
        print(cfg)

        exp_name = (
            f"{cfg['encoder_type']}_"
            f"pool{cfg['pool']}_"
            f"h{cfg['hidden_dim']}_o{cfg['output_dim']}_"
            f"layers{cfg['num_layers']}_"
            f"drop{str(cfg['dropout']).replace('.', 'p')}_"
            f"lr{cfg['lr']}_"
            f"wd{cfg['weight_decay']}_"
            f"seed{seed}"
        )

        try:
            if cfg["encoder_type"] == "gru":
                encoder = GRUEncoder(
                    input_dim=cfg["input_dim"],
                    hidden_dim=cfg["hidden_dim"],
                    output_dim=cfg["output_dim"],
                    pool=cfg["pool"],
                    dropout=cfg["dropout"],
                    num_layers=cfg["num_layers"],
                )
            elif cfg["encoder_type"] == "bigru":
                encoder = BiGRUEncoder(
                    input_dim=cfg["input_dim"],
                    hidden_dim=cfg["hidden_dim"],
                    output_dim=cfg["output_dim"],
                    pool=cfg["pool"],
                    dropout=cfg["dropout"],
                    num_layers=cfg["num_layers"],
                )
            else:
                raise ValueError(f"Unsupported encoder_type: {cfg['encoder_type']}")

            model = WordPresenceGRUModel(encoder=encoder)

            results = run_training(
                train_csv=ROOT / "dataset/task4_word_presence/splits/train_labeled.csv",
                val_csv=ROOT / "dataset/task4_word_presence/splits/val_labeled.csv",
                test_csv=ROOT / "dataset/task4_word_presence/splits/test_labeled.csv",
                feature_dir=ROOT / "dataset/task4_word_presence/pose_npy",
                out_dir=ROOT / "results/task4_word_presence/GRU_gridsearch" / exp_name,
                batch_size=cfg["batch_size"],
                num_workers=0,
                lr=cfg["lr"],
                weight_decay=cfg["weight_decay"],
                epochs=NUM_EPOCHS,
                early_stopping_patience=patience,
                model=model,
            )

            val_rank_str = (
                f"{results['val_rank_metrics']['avg_rank']:.1f}/"
                f"{results['val_rank_metrics']['num_candidates']}"
            )

            test_rank_str = (
                f"{results['test_rank_metrics']['avg_rank']:.1f}/"
                f"{results['test_rank_metrics']['num_candidates']}"
            )

            row = {
                "exp_name": exp_name,
                **cfg,
                "best_val_accuracy": results["best_val_accuracy"],
                "val_accuracy": results["best_val_metrics"]["accuracy"],
                "val_loss": results["best_val_metrics"]["loss"],
                "val_avg_rank": results["val_rank_metrics"]["avg_rank"],
                "val_num_candidates": results["val_rank_metrics"]["num_candidates"],
                "val_rank_str": val_rank_str,
                "test_accuracy": results["test_metrics"]["accuracy"],
                "test_loss": results["test_metrics"]["loss"],
                "test_avg_rank": results["test_rank_metrics"]["avg_rank"],
                "test_num_candidates": results["test_rank_metrics"]["num_candidates"],
                "test_rank_str": test_rank_str,
                "seed": seed,
            }

        except Exception as e:
            print(f"Failed config: {cfg}")
            print("Error:", e)
            row = {
                "exp_name": exp_name,
                **cfg,
                "seed": seed,
                "error": str(e),
            }

        all_results.append(row)

        pd.DataFrame([row]).to_csv(
            ROOT / results_csv_GRU_with_seeds,
            mode="a",
            header=not os.path.exists(ROOT / results_csv_GRU_with_seeds),
            index=False,
        )

df = pd.DataFrame(all_results)

if {"test_accuracy", "test_avg_rank"}.issubset(df.columns):
    display(df.sort_values(["test_accuracy", "test_avg_rank"], ascending=[False, True]))
else:
    display(df)

Total configs: 4
Sample config: {'encoder_type': 'gru', 'input_dim': 1728, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'pool': 'mean', 'num_layers': 1}

Running seed: 1990

[1/4] Running GRU model with config:
{'encoder_type': 'gru', 'input_dim': 1728, 'hidden_dim': 256, 'output_dim': 256, 'dropout': 0.0, 'lr': 0.0002, 'weight_decay': 0.0001, 'batch_size': 32, 'pool': 'mean', 'num_layers': 1}
Epoch 01 | train_loss=1.4826 | train_acc=53.64 | val_loss=1.0105 | val_acc=55.53 | val_avg_rank=46.59
Epoch 02 | train_loss=0.7872 | train_acc=64.05 | val_loss=0.8780 | val_acc=58.40 | val_avg_rank=43.22

Best validation accuracy: 58.39920948616601
Validation rank metrics: {'avg_rank': 43.21590909090909, 'avg_positive_rank': 79.03162055335969, 'num_queries': 88, 'num_candidates': 253}
Test metrics: {'loss': 0.8445332620702862, 'accuracy': 60.645933014354064}
Test rank metrics: {'avg_rank': 41.04761904761905, 'avg_positive_rank': 63.

KeyboardInterrupt: 

### converting .npy files into compressed .pt file so that dataset can be uploaded on kaggle and GPUs can be utilised

In [9]:
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

pose_dir = Path(ROOT / "dataset/task4_word_presence/pose_npy")
data = {}

for f in tqdm(pose_dir.glob("*.npy")):
    uid = f.stem
    arr = np.load(f)
    data[uid] = torch.tensor(arr, dtype=torch.float32)

torch.save(data, ROOT / "dataset/task4_word_presence/features_.pt")

2053it [00:04, 487.58it/s]


In [2]:
import pandas as pd
train_labeled_df = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task4_word_presence/splits/train_labeled.csv")
print(train_labeled_df.shape)
val_labeled_df = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task4_word_presence/splits/val_labeled.csv")
print(val_labeled_df.shape)
test_labeled_df = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/dataset/task4_word_presence/splits/test_labeled.csv")
print(test_labeled_df.shape)

(4120, 6)
(1012, 6)
(836, 6)


In [3]:
import os
from pathlib import Path
import pandas as pd

RESULT_DIR = Path("/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence")

for f in RESULT_DIR.glob("*.csv"):
    print("\n====================")
    print("FILE:", f.name)
    try:
        df = pd.read_csv(f)
        print("columns:", list(df.columns))
        print("shape:", df.shape)
    except Exception as e:
        print("Could not read:", e)


FILE: gru_gridsearch_results_with_seeds_smoke_test.csv
Could not read: Error tokenizing data. C error: Expected 13 fields in line 10, saw 23


FILE: gru_gridsearch_results.csv
columns: ['exp_name\tencoder_type\tinput_dim\thidden_dim\toutput_dim\tdropout\tlr\tweight_decay\tbatch_size\tpool\t...\tval_num_candidates\tval_rank_str\ttest_accuracy\ttest_loss\ttest_avg_rank\ttest_avg_positive_rank\ttest_num_queries\ttest_num_candidates\ttest_rank_str\tseed']
shape: (12, 1)

FILE: statpool_gridsearch_results_with_seeds.csv
Could not read: Error tokenizing data. C error: Expected 15 fields in line 10, saw 20



In [4]:
import pandas as pd

path = "/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence/gru_gridsearch_results.csv"

df = pd.read_csv(path, sep="\t")
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df.head().to_string(index=False))

shape: (12, 21)
columns: ['exp_name', 'encoder_type', 'input_dim', 'hidden_dim', 'output_dim', 'dropout', 'lr', 'weight_decay', 'batch_size', 'pool', '...', 'val_num_candidates', 'val_rank_str', 'test_accuracy', 'test_loss', 'test_avg_rank', 'test_avg_positive_rank', 'test_num_queries', 'test_num_candidates', 'test_rank_str', 'seed']
                                         exp_name encoder_type  input_dim  hidden_dim  output_dim  dropout     lr  weight_decay  batch_size    pool ...  val_num_candidates val_rank_str  test_accuracy  test_loss  test_avg_rank  test_avg_positive_rank  test_num_queries  test_num_candidates test_rank_str  seed
bigru_poolmeanstd_h256_o256_layers1_drop0p0_lr...        bigru       1728         256         256      0.0 0.0002        0.0001          32 meanstd ...                 253     15.6/253      77.990431   0.500725      18.357143               29.784689                84                  209      18.4/209    42
gru_poolmax_h256_o256_layers2_drop0p0_lr0.0002

In [5]:
import pandas as pd

path = "/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence/statpool_gridsearch_results_with_seeds.csv"

try:
    df = pd.read_csv(path, sep="\t")
except Exception as e:
    print("Normal read failed:", e)
    print("\nTrying salvage mode...")
    df = pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")

print("shape:", df.shape)
print("columns:", list(df.columns))
print(df.head().to_string(index=False))

shape: (40, 1)
columns: ['exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,pool,best_val_accuracy,val_accuracy,val_loss,test_accuracy,test_loss,seed']
                                                     exp_name,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,pool,best_val_accuracy,val_accuracy,val_loss,test_accuracy,test_loss,seed
    statpool_poolmean_h256_o256_drop0p0_lr0.0002_wd0.0001_seed1990,1728,256,256,0.0,0.0002,0.0001,32,mean,66.99604743083005,66.99604743083005,0.8479921971871094,66.38755980861244,0.8452995864398172,1990
      statpool_poolmax_h256_o256_drop0p0_lr0.0002_wd0.0001_seed1990,1728,256,256,0.0,0.0002,0.0001,32,max,67.58893280632411,67.58893280632411,0.7281346662124626,65.07177033492823,0.7385101734736319,1990
statpool_poolmeanmax_h256_o256_drop0p0_lr0.0002_wd0.0001_seed1990,1728,256,256,0.0,0.0002,0.0001,32,meanmax,62.25296442687747,62.25296442687747,0.8835885906407955,63.8755980861244,0.780573459190615,1990
  statpool

In [6]:
import pandas as pd

path = "/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence/gru_gridsearch_results_with_seeds_smoke_test.csv"

try:
    df = pd.read_csv(path, sep="\t")
except Exception as e:
    print("Normal read failed:", e)
    print("\nTrying salvage mode...")
    df = pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip")

print("shape:", df.shape)
print("columns:", list(df.columns))
print(df.head().to_string(index=False))

shape: (10, 1)
columns: ['exp_name,encoder_type,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,pool,num_layers,seed,error']
                                           exp_name,encoder_type,input_dim,hidden_dim,output_dim,dropout,lr,weight_decay,batch_size,pool,num_layers,seed,error
    gru_poolmean_h256_o256_layers1_drop0p0_lr0.0002_wd0.0001_seed1990,gru,1728,256,256,0.0,0.0002,0.0001,32,mean,1,1990,too many values to unpack (expected 2)
           gru_poolmeanstd_h256_o256_layers1_drop0p0_lr0.0002_wd0.0001_seed1990,gru,1728,256,256,0.0,0.0002,0.0001,32,meanstd,1,1990,Unsupported pool: meanstd
bigru_poolmean_h256_o256_layers1_drop0p0_lr0.0002_wd0.0001_seed1990,bigru,1728,256,256,0.0,0.0002,0.0001,32,mean,1,1990,too many values to unpack (expected 2)
       bigru_poolmeanstd_h256_o256_layers1_drop0p0_lr0.0002_wd0.0001_seed1990,bigru,1728,256,256,0.0,0.0002,0.0001,32,meanstd,1,1990,Unsupported pool: meanstd
    gru_poolmean_h256_o256_layers1_drop0p0_lr0.0002_wd0.0001_

In [7]:
import pandas as pd

path = "/Users/lokeshkumar/eMasters/EE964_Projects/results/task4_word_presence/gru_gridsearch_results.csv"
df = pd.read_csv(path, sep="\t")

best = df.sort_values(
    ["test_accuracy", "test_avg_rank"],
    ascending=[False, True]
)

print(best.head(10).to_string(index=False))

                                         exp_name encoder_type  input_dim  hidden_dim  output_dim  dropout     lr  weight_decay  batch_size    pool ...  val_num_candidates val_rank_str  test_accuracy  test_loss  test_avg_rank  test_avg_positive_rank  test_num_queries  test_num_candidates test_rank_str  seed
bigru_poolmeanstd_h256_o256_layers1_drop0p0_lr...        bigru       1728         256         256      0.0 0.0002        0.0001          32 meanstd ...                 253     15.6/253      77.990431   0.500725      18.357143               29.784689                84                  209      18.4/209    42
gru_poolmax_h256_o256_layers2_drop0p0_lr0.0002...          gru       1728         256         256      0.0 0.0002        0.0001          32     max ...                 253     16.3/253      77.511962   0.477613      15.511905               32.095694                84                  209      15.5/209    42
bigru_poolmeanstd_h256_o256_layers2_drop0p0_lr...        bigru       1728